In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('mlsa-iai-msu-lab/sci-rus-tiny')

In [ ]:
tok = model.tokenizer
MAX_TOKENS = model.max_seq_length - 48
OVERLAP = 200

In [ ]:
def split_into_sentences(text):
    ids = tok(text, truncation=False, add_special_tokens=False)["input_ids"]
    if not ids:
        return []

    boundary_ids = set()
    for p in [".", "!", "?", "…", "\n"]:
        boundary_ids |= set(tok.encode(p, add_special_tokens=False))

    sentences = []
    start = 0
    for i, tid in enumerate(ids):
        if tid in boundary_ids:
            sent = tok.decode(ids[start:i + 1], skip_special_tokens=True).strip()
            if sent:
                sentences.append(sent)
            start = i + 1

    if start < len(ids):
        sent = tok.decode(ids[start:], skip_special_tokens=True).strip()
        if sent:
            sentences.append(sent)

    return sentences


def split_into_chunks(text):
    ids = tok(text, truncation=False, add_special_tokens=False)["input_ids"]
    print(f" len(ids) : {len(ids)}")

    dot_ids = set(tok.encode(".", add_special_tokens=False))
    nl_ids  = set(tok.encode("\n", add_special_tokens=False))

    chunks = []
    start = 0

    while start < len(ids):
        end = min(start + MAX_TOKENS, len(ids))

        if end == len(ids):
            cut = end
        else:
            cut = find_last_paragraph(ids, start, end, nl_ids)
            if cut == start:
                cut = find_last_boundary(ids, start, end, dot_ids | nl_ids)
            if cut == start:
                cut = end

        chunk_ids = ids[start:cut]
        chunk_text = tok.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk_text)

        if cut == len(ids):
            break

        start = max(cut - OVERLAP, start + 1)

    return chunks


def find_last_paragraph(ids, start, end, nl_ids):
    for i in range(end - 1, start, -1):
        if ids[i] in nl_ids:
            return i + 1
    return start


def find_last_boundary(ids, start, end, boundary_ids):
    for i in range(end - 1, start - 1, -1):
        if ids[i] in boundary_ids:
            return i + 1
    return start

In [ ]:
articles = []
with open("articles_2.txt", "r", encoding="utf-8") as f:
    for block in f.read().split("\n\n"):
        lines = [l.strip() for l in block.split("\n") if l.strip()]
        if lines:
            articles.append((lines[0], lines[1:]))

In [ ]:
article_vectors = []
article_chunks = []

for title, paras in articles:
    body = " ".join(paras)
    full = title + "</s>" + body
    chunks = split_into_chunks(full)
    embs = model.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=16,
    ).astype(np.float32)

    article_vectors.append(embs)
    article_chunks.append(chunks)
    print(f"chunks: {len(chunks):3d} — {title}")

In [ ]:
def find_answer(question, top_chunks=3, top_sentences=3):
    q_emb = model.encode([question], normalize_embeddings=True)[0]

    
    all_chunks = []
    for art_idx, embs in enumerate(article_vectors):
        sims_chunk = embs @ q_emb
        for chunk_idx in range(len(sims_chunk)):
            all_chunks.append((float(sims_chunk[chunk_idx]), art_idx, chunk_idx))

    if not all_chunks:
        return []

    
    all_chunks.sort(key=lambda x: -x[0])
    best_chunks = all_chunks[:top_chunks]

    results = []
    for chunk_sim, art_idx, chunk_idx in best_chunks:
        title, _ = articles[art_idx]
        chunk_text = article_chunks[art_idx][chunk_idx]
        sents = split_into_sentences(chunk_text)

        if not sents:
            continue

        
        sent_embs = model.encode(
            sents,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=32,
        ).astype(np.float32)
        sims = sent_embs @ q_emb
        order = np.argsort(-sims)[:top_sentences]

        
        chunk_results = []
        for pos in order:
            prev_sent = sents[pos - 1] if pos - 1 >= 0 else None
            next_sent = sents[pos + 1] if pos + 1 < len(sents) else None
            chunk_results.append({
                "pos_in_chunk": int(pos),
                "prev_sentence": prev_sent,
                "sentence": sents[pos],
                "next_sentence": next_sent,
                "sim": float(sims[pos]),
            })

        results.append({
            "title": title,
            "article_idx": art_idx,
            "chunk_idx": chunk_idx,
            "chunk_sim": chunk_sim,
            "chunk_text": chunk_text,
            "sentences": chunk_results,
        })

    return results

In [ ]:
while True:
    q = input("\nquestion (or 'exit'): ").strip()
    if q.lower() == "exit":
        break
    if not q:
        continue

    hits = find_answer(q, top_chunks=5, top_sentences=3)
    for rank, h in enumerate(hits, 1):
        print(f"\n     {rank}: {h['title']}")
        print(f"chunk №{h['chunk_idx'] + 1}, chunk_sim={h['chunk_sim']:.3f}")
        for j, s in enumerate(h["sentences"], 1):
            parts = []
            if s["prev_sentence"]:
                parts.append(s["prev_sentence"])
            parts.append(s["sentence"])
            if s["next_sentence"]:
                parts.append(s["next_sentence"])
            text = " ".join(parts)
            print(f"  [{j}] sent_sim={s['sim']:.3f}: {text}")